# Always

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from collections import Counter

import torch
from IPython.display import SVG, display

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler._component_trace.resource_generalization import ConjunctionCall
from flagquantum.compiler.registers.primitives.and_compute import and_compute_ir, and_uncompute_ir
from flagquantum.core.ir.program.register_types import quantum_bit_type
from flagquantum.core.ir.program.registers import Register, RegisterSide
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit
from flagquantum.runtime.dynamic.execution import run_dynamic

## Bypass inherited controls

A composite can pass its inherited controls through an Always child
unchanged while executing the child in either control sector. The child
keeps its own declared signature and bound operands. Its adjoint
exchanges the emission functions and reverses input-only and output-only
ports.

Use this when only the middle operation of a compute–uncompute pair
needs a control. Another use is constant loading in controlled AddK.
The loads and unloads are controlled, while the middle Add runs in both
sectors. The composite author must verify the resulting controlled map.

For the locked And example the wrapper is a call-graph leaf. The source
wrapper's aggregate QEC query fails because the child is atomic. The
example reports the native child estimate and the emitted primitives
from independent queries.

In [ ]:
from flagquantum.compiler.registers.primitives.always import Always

### Example Instances

In [ ]:
bit = quantum_bit_type()
signature = Signature((
    Register("ctrl", bit, shape=(2,)),
    Register("target", bit, side=RegisterSide.RIGHT),
))
always_and = Always(
    lambda: and_compute_ir(1, 2, 3),
    lambda: and_uncompute_ir(1, 2, 3, 7),
    signature,
)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Always(And)", always_and.signature, {})])))

### Call Graph

In [ ]:
root = "Always(And)"
display(SVG(render_call_graph_svg(export_call_graph_dot(root, {}))))
print("Source graph leaf", {root: 1})
print("Owned child And estimate", query_qec_gate_cost(ConjunctionCall((1, 1))))
print("Emitted primitive counts", dict(Counter(g.name for g in always_and.emit())))

### Append compute and erase

The fourth wire starts clean. The first wire is an inherited control
and is absent from the child emission. Run both through a coherent
input and check the measured undo returns the original state.

In [ ]:
controls = (0,)
returned, operation = always_and.controlled(controls)
assert returned is controls and operation is always_and
assert all(0 not in gate.wires for gate in operation.emit())

dtype = torch.complex128
initial = torch.zeros(16, dtype=dtype)
re = torch.arange(1, 9, dtype=torch.float64)
initial[::2] = torch.complex(re, 3 - (re % 5))
initial /= initial.norm()
circuit = DynamicCircuit(4, inputs=initial, dtype=dtype)
circuit.append(FieldOperation(4, operation.emit()))
circuit.append(FieldOperation(4, operation.adjoint().emit()))
result = run_dynamic(circuit, shots=16, seed=61, strategy="batched")
outputs = result.final_states.reshape(16, 16)
torch.testing.assert_close(outputs, initial.expand_as(outputs), rtol=0, atol=3e-12)